In [1]:
import numpy as np
import nifty8 as ift
import matplotlib.pyplot as plt
from phase_II.fast_wigner_function.utils import *
from scipy.interpolate import interp1d
from phase_II.utils.helpers import *
%matplotlib tk


In [2]:
def fieldify(array, dom):
    return ift.Field(dt_(dom), array)

def Stress(xi_field):
    """

    DFT conventions:
        F[m] = sum_{n=0}^{N-1} f[n] * exp(2πi * m n / N)
        f[n] = (1/N) * sum_{m=0}^{N-1} F[m] * exp(-2πi * m n / N)

    :param xi_field:  ift.Field    The Field over real space to analyze. If harmonic, assumed to be in DFT standard order and is mapped to its real space counterpart.
    :return:
    """

    if xi_field.domain[0].harmonic:
        helper_fft = ift.FFTOperator(xi_field.domain, space=0)
        xi_time_field = helper_fft(xi_field)
    else:
        xi_time_field = xi_field


    time_space = xi_time_field.domain[0]
    h_space = time_space.get_default_codomain()

    N = time_space.size
    t_vol = time_space.scalar_dvol
    h_vol = h_space.scalar_dvol * N

    dt_step = time_space.distances[0]
    xi_time = xi_time_field.val.astype(np.complex128)
    N = len(xi_time)
    f = np.fft.fftfreq(N, d=dt_step)
    k = f.copy()
    t = np.arange(N)*dt_step
    time = np.arange(N)*dt_step

    FFT_1 = ift.FFTOperator(domain=(time_space, h_space), space=0) * (1/t_vol)
    FFT_2 = ift.FFTOperator(domain=(h_space, h_space), space=1) * (1/h_vol)

    time_cast = time[:, None]
    k_freq_cast = k[None, :]
    xi_values_cast_as_rows = xi_time[:, None]

    print("\nCalculating stress...")

    print("\t Calculating zeta plus")
    zeta_plus = fieldify(np.exp(-np.pi*k_freq_cast*1j*time_cast) * xi_values_cast_as_rows, dom=(time_space,h_space))
    print("\t Calculating zeta minus")
    zeta_minus = fieldify(np.exp(np.pi*k_freq_cast*1j*time_cast) * xi_values_cast_as_rows, dom=(time_space,h_space))

    print("\t Calculating zeta plus in Fourier space")
    tilde_zeta_plus = FFT_1(zeta_plus).val
    print("\t Calculating zeta minus in Fourier space")
    tilde_zeta_minus = FFT_1(zeta_minus).val

    print("\t Calculating Phi matrix")
    Phi_val = tilde_zeta_plus * tilde_zeta_minus.conj()  # im putting the conjugate on the MINUS zeta since I also changed FFT convention by a sign wrt. Wikipedia...
    Phi_field = ift.Field(dt_((h_space, h_space)), val=Phi_val)

    print("\t Fourier-Transforming columns of Phi matrix")
    S = FFT_2(Phi_field)
    S_mat = S.val
    print("\t ... Done")

    dk = k[1] - k[0]        # safe in FFT ordering (first step is Δk)
    dt_dual = 1.0 / (N * dk)
    t_dual = np.arange(N) * dt_dual

    return S_mat, t_dual, f



In [3]:
# -- Setup
L = 2
n_pix = 2000
dt = L/n_pix

t = ift.RGSpace(shape=(n_pix,), distances=dt)


In [4]:
real_space_white_noise = ift.from_random(t)
stress, time, freq = Stress(real_space_white_noise)


Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Fourier-Transforming columns of Phi matrix
	 ... Done


In [5]:
smooth=True
if smooth:
    stress_matrix = gaussian_filter(stress.real, sigma=(.5,.5))   # sigma ~ 0.5..3 blur radius in pixels

visualize_stress(stress_matrix, rows=freq, cols=time, smooth=False)
print(np.mean(stress.real), np.std(stress.real))

		Rows must be in ascending order for visualization purposes but they are not, assuming a priori standard DFT order and moving DC to the middle
1.006647251904957 45.00275748955215


In [6]:
S_mat_collection = []
for _ in range(10):
    real_space_white_noise = ift.from_random(t)
    stress, _, _ = Stress(real_space_white_noise)
    S_mat_collection.append(np.array(stress.real))



Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Fourier-Transforming columns of Phi matrix
	 ... Done

Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Fourier-Transforming columns of Phi matrix
	 ... Done

Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Fourier-Transforming columns of Phi matrix
	 ... Done

Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Fourier-Transforming columns of Phi matrix
	 ... Done

Calculating stress...
	 Calculating zeta plus
	 Calcula

In [8]:
S_mat_avg = np.mean(S_mat_collection, axis=0)

smooth=True
if smooth:
    stress_matrix = gaussian_filter(S_mat_avg, sigma=5)   # sigma ~ 0.5..3 blur radius in pixels

visualize_stress(stress_matrix, rows=freq, cols=time, smooth=True)

S_mat_std = np.std(S_mat_collection, axis=0)
print("Mean and mean std of original: ", np.mean(S_mat_avg), np.mean(S_mat_std))
print("Mean and mean std of smoothed version: ", np.mean(stress_matrix), np.mean(stress_matrix))

		Rows must be in ascending order for visualization purposes but they are not, assuming a priori standard DFT order and moving DC to the middle
Mean and mean std of original:  1.0089779392473073 39.22660160572426
Mean and mean std of smoothed version:  1.008977939247307 1.008977939247307


In [ ]:
t_star = 1.8
real_time_xi_peak = xi_field(case=1, N=n_pix, peak_frequency=t_star, peak_amplitude=100, omegas=time)
print("\tThink of peak at time t_star!")

real_xi_peak_field = fieldify(real_time_xi_peak, dom=t)

stress, time, freq = Stress(real_xi_peak_field)
visualize_stress(stress, rows=freq, cols=time)

max_of_matrix_y, max_of_matrix_x  = np.argwhere(stress.real == np.max(stress.real))[0]  # first row corresponding to a max

print("max time of stress: ", time[max_of_matrix_x])

In [ ]:
f_star = -15.5
fourier_xi_peak = xi_field(case=1, N=n_pix, peak_frequency=f_star, peak_amplitude=100, omegas=freq)
real_xi_peak_from_fourier = np.fft.ifft(fourier_xi_peak, n=n_pix)

real_xi_peak_field_from_fourier = fieldify(real_xi_peak_from_fourier, dom=t)

In [ ]:
stress, time, freq = Stress(real_xi_peak_field_from_fourier)
visualize_stress(stress.real, rows=freq, cols=time)

max_of_matrix_y, max_of_matrix_x  = np.argwhere(stress.real == np.max(stress.real))[0]  # first row corresponding to a max

print(freq[max_of_matrix_y])

In [ ]:
# Gaussian wave paket
t_star_idx = 1200
f_star_idx = 220

t_star = time[t_star_idx]
f_star = freq[f_star_idx]

print("t and f star are: ", t_star, f_star)

xi_field_val_real_space =  1e6*np.exp(-1/2 * (time-t_star)**2 / 0.01**2) * np.exp(2*np.pi *1j* f_star * time)

xi_field_val_fourier_space = np.fft.fft(xi_field_val_real_space, n=n_pix)

xi_field_gaussian_wave_packet_real_space = fieldify(xi_field_val_real_space, dom=t)

In [ ]:
fig, axs = plt.subplots(2,1)
axs[0].plot(time, xi_field_val_real_space.real, label="real part")
axs[1].plot(np.fft.fftshift(freq), np.fft.fftshift(xi_field_val_fourier_space.real) , label="real part")
axs[0].set_xlabel("time in seconds")
axs[1].set_xlabel("Frequency in Hz")
axs[0].set_ylabel(r"Real $\xi$")
axs[1].set_ylabel(r"Harmonic $\xi$")
plt.tight_layout()
plt.show()

In [ ]:
stress, time, freq = Stress(xi_field_gaussian_wave_packet_real_space)
visualize_stress(stress.real, rows=freq, cols=time, tl=" : Gaussian Wave packet")

In [ ]:
# Multiple spikes in t ...

real_time_xi_peak = np.zeros(n_pix)

peak_pos = [1500, 1550]
# peak_pos = np.arange(100, 2000, 1500)
print("peak pos: ", peak_pos)

real_time_xi_peak[peak_pos] = 1e3

print("t stars @ : ",  time[peak_pos])

xi_field = fieldify(real_time_xi_peak, dom=t)

stress, time, freq = Stress(xi_field)
visualize_stress(stress, rows=freq, cols=time, tl=" : Interference term from the Wigner function!")


In other words, if we have multiple peaks in fourier space, what will our stress function look like? Note that the number of interference terms depends on the number of peaks and the period of the oscillations they produce (oscillations in time if peaks in frequency) depend on the distance of the peaks in frequency/harmonic space

In [ ]:
# Multiple spikes in t ...

harmonic_space_xi_peak = np.zeros(n_pix)

peak_pos = [1500, 1550, 200, 400, 500]
freq_pos = freq[peak_pos]
# peak_pos = np.arange(100, 2000, 1500)
print("peak pos: ", peak_pos)
print("Corresponding to f stars: ", freq_pos)

harmonic_space_xi_peak[peak_pos] = 1e3

real_time_xi_peak = np.fft.ifft(harmonic_space_xi_peak, n=n_pix)

xi_field = fieldify(real_time_xi_peak, dom=t)

stress, time, freq = Stress(xi_field)

remove_interference_terms = False
if remove_interference_terms:
    stress = stress.real.copy()
    stress[stress<0] = 0

visualize_stress(stress, rows=freq, cols=time, tl=" : Interference term from the Wigner function!")
